# WorldSim v3 — Task E Fix & Retrain
Fix action_id validation bug, regenerate Task E, reassemble, retrain dual models


## 1. Verify Bug Fix


In [ ]:
from pathlib import Path
import json
import os
import sys

from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config" / "generation.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
    if not (REPO_ROOT / "config" / "generation.yaml").exists():
        raise RuntimeError("Run this notebook from the repository root or notebooks directory")
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
load_dotenv(REPO_ROOT / ".env")

from scripts.validate_data import _validate_option_ids

record_plain = {"action_options": ["flee", "hide", "confront", "warn", "freeze"]}
payload_valid = {"action_id": 3}
payload_invalid = {"action_id": 7}

v1 = _validate_option_ids("E", payload_valid, record_plain)
v2 = _validate_option_ids("E", payload_invalid, record_plain)
assert v1 == [], f"Expected no violations for action_id=3 in 5 options, got {v1}"
assert v2 == ["invalid_action_id"], f"Expected violation for action_id=7, got {v2}"

record_dict = {"action_options": [{"id": 0, "ko": "a"}, {"id": 1, "ko": "b"}]}
v3 = _validate_option_ids("E", {"action_id": 1}, record_dict)
v4 = _validate_option_ids("E", {"action_id": 5}, record_dict)
assert v3 == [], f"Dict options broken: {v3}"
assert v4 == ["invalid_action_id"], f"Dict options broken: {v4}"

print("✅ Bug fix verified: plain string options now work correctly")

from scripts.generate_data import _build_jobs_from_catalogs, load_catalogs, load_generation_config, parse_and_validate

settings = load_generation_config(REPO_ROOT / "config")
catalogs = load_catalogs(REPO_ROOT / "config")
jobs = _build_jobs_from_catalogs(catalogs, settings, system_prompt="test", seed=42, schema_version=3)
e_job = next(job for job in jobs if job["task"] == "E")

test_output = json.dumps(
    {
        "action_id": 3,
        "confidence": 0.72,
        "hint": "High caution still drives a sharp warning rather than a reckless charge.",
        "personality_reasoning": "harm_avoidance",
        "temperament_factor": "choleric_warning_impulse",
    },
    ensure_ascii=False,
)
validated, error = parse_and_validate(test_output, e_job, settings)
assert error is None, f"Full pipeline still broken: {error}"
print(f"✅ Full pipeline verified: Task E validation passes -> {validated}")


## 2. Generate Task E


In [ ]:
import time

from scripts.generate_data import generate_dataset, load_batch_plan

batch_plan = load_batch_plan(REPO_ROOT, batch_id="batch_v3_03_task_e_fix")
target = sum((batch_plan.get("task_counts") or batch_plan.get("tasks", {})).values())

print(f"Generating Task E: {target} requests", flush=True)
print("Estimated cost: ~$1", flush=True)
print("=" * 50, flush=True)

started = time.perf_counter()
result = generate_dataset(REPO_ROOT, batch_plan=batch_plan)
elapsed = time.perf_counter() - started

print()
print("=" * 50)
print(f"Task E generation complete in {elapsed:.0f}s ({elapsed / 60:.1f} min)")
result


## 3. Validate Task E


In [ ]:
from scripts.common import read_jsonl
from scripts.validate_data import validate_file

raw_dir = REPO_ROOT / "data" / "raw" / "batch_v3_03_task_e_fix"
latest = max(raw_dir.glob("*.jsonl"), key=lambda p: p.stat().st_mtime)
validated_dir = REPO_ROOT / "data" / "validated" / "batch_v3_03_task_e_fix"

report = validate_file(input_path=latest, validated_dir=validated_dir, repo_root=REPO_ROOT)
print(f"Passed: {report.get('passed', 0)}")
print(f"Failed: {report.get('failed', 0)}")

passed_path = validated_dir / "passed.jsonl"
passed_rows = read_jsonl(passed_path)
print(f"Task E validated rows: {len(passed_rows)}")
assert len(passed_rows) >= 150, f"Too few Task E rows: {len(passed_rows)}"
print("✅ Validation complete")


## 4. Merge into Existing v3 Data


In [ ]:
import shutil
from collections import Counter

from scripts.common import read_jsonl, write_jsonl

logic_path = REPO_ROOT / "data" / "validated" / "batch_v3_01_english_logic" / "passed.jsonl"
new_tasks_path = REPO_ROOT / "data" / "validated" / "batch_v3_02_new_tasks" / "passed.jsonl"
task_e_path = REPO_ROOT / "data" / "validated" / "batch_v3_03_task_e_fix" / "passed.jsonl"

logic_rows = read_jsonl(logic_path)
new_rows = read_jsonl(new_tasks_path)
task_e_rows = read_jsonl(task_e_path)

backup = logic_path.with_suffix(".jsonl.bak_before_task_e")
if not backup.exists() and logic_path.exists():
    shutil.copy2(logic_path, backup)
    print(f"Backed up original to {backup.name}")

merged_logic = logic_rows + task_e_rows
write_jsonl(logic_path, merged_logic)

task_counts = Counter(row.get("task", "?") for row in merged_logic)
print()
print(f"Merged logic batch: {len(merged_logic)} rows")
for task, count in sorted(task_counts.items()):
    print(f"  Task {task}: {count}")
print()
print(f"Reference new-task rows still available: {len(new_rows)}")
print("✅ Task E merged into batch_v3_01 passed.jsonl")


## 5. Reassemble v3 Dataset


In [ ]:
from collections import Counter

from scripts.assemble_v3_dataset import assemble_v3_dataset
from scripts.common import read_jsonl

V3_DATASET_ID = "worldsim-v3-mix"

assembly_result = assemble_v3_dataset(
    REPO_ROOT,
    dataset_id=V3_DATASET_ID,
    dev_ratio=0.1,
    seed=42,
)

manifest = assembly_result.manifest
print("=" * 60)
print("  V3 DATASET REASSEMBLED (with Task E)")
print("=" * 60)
print(f"  Train: {manifest['output']['train']}")
print(f"  Dev:   {manifest['output']['dev']}")
print(f"  Total: {manifest['output']['total']}")

v3_final_dir = REPO_ROOT / "data" / "final" / V3_DATASET_ID
v3_train = read_jsonl(v3_final_dir / "train.jsonl")
task_counts = Counter(row.get("task", "?") for row in v3_train)
print()
print("  Task distribution (train):")
for task, count in sorted(task_counts.items()):
    marker = "⭐" if task == "E" else "  "
    print(f"  {marker} {task}: {count}")
assert "E" in task_counts and task_counts["E"] > 0, "Task E still missing!"
print()
print(f"✅ Task E present: {task_counts['E']} rows")


## 6. Convert to Training Format


In [ ]:
import math

from scripts.common import read_jsonl, write_jsonl
from scripts.convert_mixed_final_to_training_format import convert_mixed_final_to_training_format
from scripts.curriculum_order_v3 import curriculum_order_v3

V3_TRAINING_DIR = REPO_ROOT / "data" / "training" / V3_DATASET_ID

conversion_result = convert_mixed_final_to_training_format(
    repo_root=REPO_ROOT,
    input_train=v3_final_dir / "train.jsonl",
    input_dev=v3_final_dir / "dev.jsonl",
    source_manifest=assembly_result.manifest_path,
    output_dir=V3_TRAINING_DIR,
    dataset_id=V3_DATASET_ID,
)

train_converted = conversion_result.train_output
dev_converted = conversion_result.dev_output
train_curriculum = V3_TRAINING_DIR / "train_curriculum.jsonl"
converted_rows = read_jsonl(train_converted)
ordered_rows = curriculum_order_v3(converted_rows, seed=42)
write_jsonl(train_curriculum, ordered_rows)

train_count = len(ordered_rows)
dev_count = conversion_result.dev_count


def compute_three_epoch_steps(row_count: int, *, batch_size: int = 1, grad_accum: int = 8, epochs: int = 3) -> int:
    effective_batch = max(1, batch_size * grad_accum)
    steps_per_epoch = max(1, math.ceil(row_count / effective_batch))
    return steps_per_epoch * epochs


max_steps_v3 = compute_three_epoch_steps(train_count)

print(f"Train: {train_count}, Dev: {dev_count}")
print(f"Max steps (3 epochs): {max_steps_v3}")
print("✅ Conversion complete")


## 7. Retrain 0.8B


In [ ]:
import time
from datetime import UTC, datetime

from training.lib.qlora_smoke import SmokeRunConfig, run_baseline_or_raise

MODEL_08B = "Qwen/Qwen3.5-0.8B-Base"
RUN_ID_08B = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%SZ")
OUTPUT_08B = REPO_ROOT / "outputs" / "baseline" / "worldsim-v3-08b" / RUN_ID_08B

cfg_08b = SmokeRunConfig(
    run_mode="baseline",
    model_name=MODEL_08B,
    train_file=train_curriculum,
    dev_file=dev_converted,
    output_dir=OUTPUT_08B,
    max_steps=max_steps_v3,
    max_train_samples=0,
    max_eval_samples=0,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_steps=100,
    save_steps=100,
    save_total_limit=1,
    require_qlora=True,
    seed=42,
    dry_run=False,
)

print("Training 0.8B on v3 data (with Task E)...", flush=True)
started_08b = time.perf_counter()
result_08b = run_baseline_or_raise(cfg_08b)
elapsed_08b = time.perf_counter() - started_08b

print()
print("=" * 60)
print(f"  0.8B COMPLETE ({elapsed_08b / 60:.1f} min)")
print("=" * 60)
print(f"  train_loss: {result_08b.train_loss}")
print(f"  eval_loss:  {result_08b.eval_loss}")


## 8. Retrain 2B


In [ ]:
MODEL_2B = "Qwen/Qwen3.5-2B-Base"
RUN_ID_2B = datetime.now(UTC).strftime("run-%Y%m%dT%H%M%SZ")
OUTPUT_2B = REPO_ROOT / "outputs" / "baseline" / "worldsim-v3-2b" / RUN_ID_2B

cfg_2b = SmokeRunConfig(
    run_mode="baseline",
    model_name=MODEL_2B,
    train_file=train_curriculum,
    dev_file=dev_converted,
    output_dir=OUTPUT_2B,
    max_steps=max_steps_v3,
    max_train_samples=0,
    max_eval_samples=0,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    logging_steps=10,
    eval_steps=100,
    save_steps=100,
    save_total_limit=1,
    require_qlora=True,
    seed=42,
    dry_run=False,
)

print("Training 2B on v3 data (with Task E)...", flush=True)
started_2b = time.perf_counter()
result_2b = run_baseline_or_raise(cfg_2b)
elapsed_2b = time.perf_counter() - started_2b

print()
print("=" * 60)
print(f"  2B COMPLETE ({elapsed_2b / 60:.1f} min)")
print("=" * 60)
print(f"  train_loss: {result_2b.train_loss}")
print(f"  eval_loss:  {result_2b.eval_loss}")


## 9. GGUF Conversion


In [ ]:
import shutil
import subprocess

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

LLAMA_QUANTIZE = REPO_ROOT / "tools" / "llama.cpp" / "build" / "bin" / "llama-quantize"
CONVERT_SCRIPT = REPO_ROOT / "tools" / "llama.cpp" / "convert_hf_to_gguf.py"
GGUF_DIR = REPO_ROOT / "artifacts" / "gguf"


def merge_and_convert_gguf(model_name: str, output_dir: str | Path, gguf_name: str) -> Path:
    output_dir = Path(output_dir)
    adapter_dir = output_dir / "adapter"
    merged_dir = output_dir / "merged_bf16"
    fp16_gguf = output_dir / f"{gguf_name}-f16.gguf"
    q4_gguf = output_dir / f"{gguf_name}-q4_k_m.gguf"

    if not (merged_dir / "config.json").exists():
        print(f"  Merging LoRA for {model_name}...", flush=True)
        base = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        merged = PeftModel.from_pretrained(base, str(adapter_dir)).merge_and_unload()
        merged_dir.mkdir(parents=True, exist_ok=True)
        merged.save_pretrained(str(merged_dir))
        tokenizer.save_pretrained(str(merged_dir))
        del merged, base
        torch.cuda.empty_cache()
        print("  Merged ✅", flush=True)

    if not fp16_gguf.exists():
        print("  Converting to GGUF FP16...", flush=True)
        result = subprocess.run(
            [sys.executable, str(CONVERT_SCRIPT), str(merged_dir), "--outfile", str(fp16_gguf), "--outtype", "f16"],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"GGUF conversion failed: {result.stderr[-500:]}")
        print(f"  FP16: {fp16_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)

    if not q4_gguf.exists():
        print("  Quantizing to Q4_K_M...", flush=True)
        result = subprocess.run(
            [str(LLAMA_QUANTIZE), str(fp16_gguf), str(q4_gguf), "Q4_K_M"],
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            raise RuntimeError(f"Quantization failed: {result.stderr[-500:]}")
        print(f"  Q4_K_M: {q4_gguf.stat().st_size / 1e6:.0f} MB ✅", flush=True)
    return q4_gguf


print("=== 0.8B GGUF ===", flush=True)
gguf_08b = merge_and_convert_gguf(MODEL_08B, result_08b.output_dir, "worldsim-v3-qwen3.5-0.8b")
final_08b = GGUF_DIR / "worldsim-v3-qwen3.5-0.8b-q4_k_m.gguf"
shutil.copy2(gguf_08b, final_08b)

print()
print("=== 2B GGUF ===", flush=True)
gguf_2b = merge_and_convert_gguf(MODEL_2B, result_2b.output_dir, "worldsim-v3-qwen3.5-2b")
final_2b = GGUF_DIR / "worldsim-v3-qwen3.5-2b-q4_k_m.gguf"
shutil.copy2(gguf_2b, final_2b)

print()
print("=" * 60)
print(f"  0.8B: {final_08b.name} ({final_08b.stat().st_size / 1e6:.0f} MB)")
print(f"  2B:   {final_2b.name} ({final_2b.stat().st_size / 1e6:.0f} MB)")
print(f"  Combined: {(final_08b.stat().st_size + final_2b.stat().st_size) / 1e6:.0f} MB")


## 10. Final Summary


In [ ]:
from scripts.common import read_jsonl

print("=" * 70)
print("  WORLDSIM V3 — TASK E FIX COMPLETE")
print("=" * 70)
print()
print("  [Bug Fix]")
print("    _validate_option_ids: plain string fallback added")
print()
print("  [Task E Regeneration]")
print(f"    Generated: {len(read_jsonl(task_e_path))} rows")
print()
print("  [Dataset]")
print(f"    Train: {train_count}, Dev: {dev_count}")
print(f"    Tasks: {len(task_counts)} (including E: {task_counts.get('E', 0)})")
print()
print("  [Training]")
print(f"    0.8B: train_loss={result_08b.train_loss:.4f}, eval_loss={result_08b.eval_loss:.4f}")
print(f"    2B:   train_loss={result_2b.train_loss:.4f}, eval_loss={result_2b.eval_loss:.4f}")
print()
print("  [GGUF]")
print(f"    0.8B: {final_08b.stat().st_size / 1e6:.0f} MB")
print(f"    2B:   {final_2b.stat().st_size / 1e6:.0f} MB")
print("=" * 70)
